# o3/o4-mini 函数调用指南

## 简介 
o3/o4-mini 是我们 o 系列模型中的最新成员，这些模型经过训练，会在回答之前进行更长时间的思考。它们是我们迄今发布的最智能的模型，在工具调用能力上较 o1/o3-mini 有了显著进步。这些模型经过训练，能够在思维链（CoT）中原生地使用工具，从而在何时以及如何使用工具方面具备更强的推理能力。我们已经发布了一份关于如何通过 responses API 使用这些模型[调用函数](https://cookbook.openai.com/examples/reasoning_function_calls)的指南，本指南在此基础上，告诉你如何让这些模型发挥最佳的函数调用性能。

## 提升函数调用性能的提示指导
为了充分利用 o3/o4-mini 模型背后的函数调用智能，我们建议在开发者提示和函数描述中遵循一些最佳实践。

### 关于推理模型的开发者提示、系统提示和函数描述的简要说明
我们引入了开发者消息，以便明确告诉推理模型某条指令来自开发者。在 o 系列模型中，开发者提供的任何系统消息都会在内部自动转换为开发者消息。从实用角度看，你可以把开发者提示视为传统系统提示的等价物——但为了清晰和准确，本指南将所有此类指令统称为开发者提示/消息。

本文档中提到的函数描述，指的是 API 请求的 tool 参数中每个函数对象的 description 字段里的说明文字。该描述告诉模型何时以及如何使用该函数。以下是我们函数调用[文档](https://platform.openai.com/docs/guides/function-calling)中的一个示例：

```
tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]
```
这里，`"Get current temperature for provided coordinates in celsius."` 就是函数描述。

明确了这些定义之后，我们就可以开始介绍最佳实践了。



### 通过开发者消息设定上下文
1. 总体上下文：与通用的提示工程最佳实践一致，角色提示有助于设定基本行为和语气，并概述可执行的操作范围。例如：
```
You are an AI retail agent.

As a retail agent, you can help users cancel or modify pending orders, return or exchange delivered orders, modify their default user address, or provide information about their own profile, orders, and related products.
```
2. 函数调用顺序：o3/o4-mini 经过训练，能够借助工具完成目标。然而，它们可能会弄错工具调用的顺序。为防止这种情况，建议明确列出完成特定任务的顺序。例如，为了防止编程智能体在尚不存在的目录中创建文件，通常加上下面这句就足够了：
```
check to see if directories exist before making files
```
对于高频且定义明确的任务，我们可以通过明确列出要调用的函数序列来使其更加稳健，例如：
```
To Process a refund for a delivered order, follow the following steps:
1. Confirm the order was delivered. Use: `order_status_check`
2. Check the refund eligibility policy. Use: `refund_policy_check`
3. Create the refund request. Use: `refund_create`
4. Notify the user of refund status. Use: `user_notify`
```

3. 界定何时使用工具：向模型明确何时应该、何时不应该调用某些工具是很有帮助的。这既可以在开发者提示层面完成，也可以在工具描述层面完成。下面是一个开发者提示示例：
```
Be proactive in using tools to accomplish the user's goal. If a task cannot be completed with a single step, keep going and use multiple tools as needed until the task is completed. Do not stop at the first failure. Try alternative steps or tool combinations until you succeed.

- Use tools when:
  - The user wants to cancel or modify an order.
  - The user wants to return or exchange a delivered product.
  - The user wants to update their address or contact details.
  - The user asks for current or personalized order or profile info.

- Do not use tools when:
  - The user asks a general question like “What’s your return policy?”
  - The user asks something outside your retail role (e.g., “Write a poem”).

If a task is not possible due to real constraints (For example, trying to cancel an already delivered order), explain why clearly and do not call tools blindly.
```

### 函数描述
函数的描述应当说明何时调用它以及如何构造其参数。

函数描述是说明函数何时应被调用、参数应如何构造的理想位置。它充当推理模型与工具 API 之间持久的接口契约。

一般来说，函数描述定义了函数做什么、如何调用它；开发者指令则为使用这些工具的智能体提供指导。因此，如果有多个工具可用于类似目的，开发者可以在指令中对它们加以区分。如果智能体工作流要求按特定顺序使用工具，或者要求频繁或节制地使用某些工具，这些内容也应写入开发者指令中。

结构良好的描述可以通过尽早锚定关键标准和参数要求来提高准确性并减少误调用。它还让开发者可以在开发者提示之外、在更靠近工具定义本身的地方编写"主动性"控制启发式规则。 

1. 使用标准：就像可以通过开发者提示细化函数调用的主动性一样，你也可以在函数描述层面进一步细化函数的调用方式。下面是一个 file_create 函数的示例：
```
Creates a new file with the specified name and contents in a target directory. This function should be used when persistent storage is needed and the file does not already exist.
- Only call this function if the target directory exists. Check first using the `directory_check` tool.  
- Do not use for temporary or one-off content—prefer direct responses for those cases.  
- Do not overwrite existing files. Always ensure the file name is unique.
- Do not overwrite existing files.  
  If replacement is intended and confirmed, use `file_delete` followed by `file_create`, or use `file_update` instead.
```
2. 少样本提示：虽然推理模型从少样本提示中获得的收益不如非推理模型那么多，但我们发现少样本提示可以提升工具调用性能，尤其是在模型难以准确构造函数参数时。例如，下面是一个作为工具描述传入的 grep 工具描述示例： 

```
Use this tool to run fast, exact regex searches over text files using the `ripgrep` engine.


- Always escape special regex characters: ( ) [ ] { } + * ? ^ $ | . \\
- Use `\\` to escape any of these characters when they appear in your search string.
- Do NOT perform fuzzy or semantic matches.
- Return only a valid regex pattern string.

Examples:
Literal            -> Regex Pattern         
function(          -> function\\(           
value[index]       -> value\\[index\\]      
file.txt           -> file\\.txt            
user|admin         -> user\\|admin          
path\to\file       -> path\\\\to\\\\file     
```

3. 关键规则前置，尽量减少干扰：注意在上面的示例中，转义特殊字符的指令几乎是模型读到的第一件事。一个**更差**的写法是：
```
Performs a fast regex-based text search that looks for exact pattern matches within files or entire directories, leveraging the ripgrep tool for high-speed scanning.
Output follows ripgrep formatting and can optionally display line numbers and matched lines.
To manage verbosity, results are limited to a maximum of 50 hits.
You can fine-tune the search by specifying inclusion or exclusion rules based on file types or path patterns.
This method is ideal when searching for literal text snippets or specific regular expressions.
It offers more accuracy than semantic methods when the goal is to locate a known string or structure.
It’s generally recommended over semantic search when you’re looking for a specific identifier—such as a function name, variable, or keyword—within a defined set of directories or file types.
```

这个写法效果较差，因为提示中的大部分内容都不是规定性的，而且关于如何构造参数的最重要规则没有放在最显眼的位置。在针对这个 ripgrep 工具的工具调用准确率评估中，前一个提示的得分比上面这个高 6%。





### 防范函数调用幻觉 
我们注意到，o3 模型可能比其他模型更容易产生幻觉。这些幻觉可能表现为模型承诺在后台调用工具但实际上并没有调用，或者承诺在之后的轮次中调用工具等。在这类情况下，在以下几个方面给出明确说明，有助于尽量减少此类幻觉：

1. 明确的指令：明确指示模型避免常见的幻觉，例如在无法做到时承诺未来会调用函数。

```
Do NOT promise to call a function later. If a function call is required, emit it now; otherwise respond normally.
```

2. 尽早发现错误参数：
将 `strict` 设置为 `true` 可以确保函数调用可靠地遵循[函数 schema](https://platform.openai.com/docs/guides/function-calling?api-mode=responses#strict-mode)。我们建议尽可能开启它。

如果你的参数还有其他复杂的格式要求（例如必须是有效的 python 代码等），加入下面的指令可以提醒模型注意预期格式。 

```
Validate arguments against the format before sending the call; if you are unsure, ask for clarification instead of guessing.
```

3. 关于偷懒行为的另一点说明：
我们注意到 o3 偶尔会出现偷懒行为，例如声称没有足够时间完成任务、承诺稍后单独跟进，或者即使被明确要求提供更多细节，仍然给出简短的回答。我们发现以下措施有助于改善这种行为：

    a. 为无关话题开启新对话：  
       切换到新的或无关的话题时，请开启一个全新的对话线程，而不是在同一上下文中继续。这有助于模型专注于当前主题，避免受到之前无关上下文的影响——这种影响有时会导致不完整或偷懒的回答。例如，如果你之前在讨论代码调试，现在想询问文档编写的最佳实践，而这并不需要之前的对话上下文，那么请开启新对话，以确保清晰和专注。

    b. 当列表过长时，丢弃无关的历史工具调用/输出，并在用户消息中将其总结为上下文：  
       如果对话历史中包含一长串不再相关的历史工具调用或输出，请将它们从上下文中移除，转而在用户消息中提供重要信息的简明摘要。这样可以让上下文保持在可控范围内，并确保模型只接触到最相关的信息。例如，如果你有一长串工具输出，可以总结关键结果，并在下一条消息中只包含该摘要。

    c. 我们一直在持续改进模型，预计会在未来的版本中解决这个问题。


### 避免思维链提示
由于这些模型是推理模型，会生成内部思维链，因此不需要显式提示它们在工具调用之间进行规划和推理。所以，开发者不应试图通过要求模型做更充分的规划来在每次函数调用前诱导额外的推理。要求推理模型进行更多推理，实际上可能会损害性能。 

关于推理摘要的简短补充：模型在调用工具之前会输出推理 token。不过，这些 token 并不总会附带摘要，因为我们的推理摘要需要达到一定数量的实质性推理 token 才会生成。


# Responses API

### 使用推理条目提升性能
我们发布了一份 [cookbook](https://cookbook.openai.com/examples/responses_api/reasoning_items)，详细介绍了使用 responses API 的好处。这里也值得重申其中几个要点。o3/o4-mini 在训练时，其内部推理会在单个轮次内的多次工具调用之间保留。因此，在推理过程中于工具调用之间保留这些推理条目，会带来更高的智能和性能，体现为在何时以及如何调用工具方面做出更好的决策。Responses 允许你保留这些推理条目（可以由我们维护，如果你不希望我们处理状态管理，也可以由你自己通过加密内容来维护），而 Chat Completion 则不支持。切换到 responses API 并让模型在函数调用之间访问推理条目，是尽可能提升函数调用性能的最简单方法。下面是 cookbook 中的示例，为方便起见在此转载，展示了如何使用 `encrypted_content` 回传推理条目，而我们这边不会保留任何状态：


In [2]:
from openai import OpenAI
import requests
import json
client = OpenAI()


def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

context = [{"role": "user", "content": "What's the weather like in Paris today?"}]

response = client.responses.create(
    model="o3",
    input=context,
    tools=tools,
    store=False,
    include=["reasoning.encrypted_content"] # Encrypted chain of thought is passed back in the response
)


context += response.output # Add the response to the context (including the encrypted chain of thought)
tool_call = response.output[1]
args = json.loads(tool_call.arguments)



result = get_weather(args["latitude"], args["longitude"])

context.append({                               
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model="o3",
    input=context,
    tools=tools,
    store=False,
    include=["reasoning.encrypted_content"]
)

print(response_2.output_text)

The current temperature in Paris is about 18.8 °C.


## 使用托管工具的智能体体验。 
Responses API 支持一组托管/内置工具。我们最近还在 responses API 中加入了[新的工具和功能](https://openai.com/index/new-tools-and-features-in-the-responses-api/)，让构建连接外部服务的智能体应用变得更加容易；借助 Responses API 中的内置工具，开发者只需一次 API 调用就能创建能力更强的智能体。

你可以在同一个会话中混合使用托管工具和自定义工具。这带来了强大的组合模式，但也使得工具路由的清晰性变得至关重要。以下是几条具体建议：

1. 在开发者提示中明确界定工具的使用范围：如果多个工具可以承担类似的角色（例如 python 工具和自定义计算器），请告诉模型优先使用哪个工具以及何时使用。这可以减少歧义、提高准确性，并避免过度使用或使用不足。：
```
You are a helpful research assistant with access to the following tools:
- python tool: for any computation involving math, statistics, or code execution
- calculator: for basic arithmetic or unit conversions when speed is preferred

Always use the python tool for anything involving logic, scripts, or multistep math. Use the calculator tool only for simple 1-step math problems.
```

2. 说明何时内部知识不够用：尽管 o3/o4-mini 模型常常能够独立解决任务，但工具可能提供更可靠的答案。使用系统提示引导模型在更适合使用工具时，不要"试图自己解决"。

```
You have access to a `code_interpreter`. Always prefer using `code_interpreter` when a user asks a question involving:
- math problems
- data analysis
- generating or executing code
- formatting or transforming structured text

Avoid doing these directly in your own response. Always use the tool instead.
```

3. 由于开发者提示充当的是一个集中、持久的契约，当我们希望将托管工具与你的自定义函数混合使用时，请在这里写清楚工具的决策边界，包括功能覆盖的重叠、置信度预期或回退行为：

```
Use `python` for general math, data parsing, unit conversion, or logic tasks that can be solved without external lookup—for example, computing the total cost from a list of prices.

Use `calculate_shipping_cost` when the user asks for shipping estimates, as it applies business-specific logic and access to live rate tables. Do not attempt to estimate these using the `python` tool.

When both could be used (e.g., calculating a delivery fee), prefer `calculate_shipping_cost` for accuracy and policy compliance. Fall back to `python` only if the custom tool is unavailable or fails.
```

4. 关于 MCP 的更多内容：我们有一份更详细的使用 MCP 工具最佳实践[指南](https://cookbook.openai.com/examples/mcp/mcp_tool_guide)，但为了完整起见，这里再重申几条高层次的准则（这些准则并非 o3/o4-mini 特有，但仍然适用）。

* 过滤工具，避免负载膨胀：利用 allowed_tools 参数只使用必要的工具，节省不必要的上下文：由于你并不总是需要 MCP 服务器返回的所有工具，可以通过 allowed_tools 字段只保留必要的工具。


```
    "tools": [
        {
            "type": "mcp",
            "server_label": "gitmcp",
            "server_url": "https://gitmcp.io/openai/tiktoken",
            "allowed_tools": ["search_tiktoken_documentation", "fetch_tiktoken_documentation"],
            "require_approval": "never"
        }
```

* 通过缓存降低延迟，并将推理模型留给高复杂度任务：确保回传 `mcp_list_tools` 或包含 `previous_response_id`，这样 API 就不需要一次又一次地重复导入工具列表。
* 将 MCP 与其他工具结合使用：你可以将 MCP 与其他托管工具以及你自定义的函数混合使用。如果混合使用工具，最好在整体开发者提示中定义决策边界，并明确说明何时使用某个工具而不是另一个。[这里](https://cookbook.openai.com/examples/mcp/mcp_tool_guide#using-mcp-with-other-tools)是 MCP 工具指南中的一个很好的示例。


### 常见问题（FAQ）

**问：多少个函数算太多？**

**答：** 对于 o3 和 o4-mini 模型，函数数量没有硬性上限，但根据训练数据分布和观察到的模型行为，确实存在一些实用的指导。截至 2025 年 5 月，任何工具数量少于约 100 个、每个工具参数少于约 20 个的配置都被视为处于训练分布之内，其表现应在预期的可靠性范围内。性能仍然取决于你的提示设计和任务复杂度。 

即使从技术上讲你处于训练分布之内，更多的工具也可能带来歧义或混乱。以下是关键考虑因素：

* 函数描述的清晰度变得至关重要：如果多个工具的用途重叠或描述含糊，模型可能会调用错误的工具，或者犹豫不决、一个都不调用。

* 工具列表的大小会影响延迟和推理深度：列表越长，模型在推理阶段需要解析的选项就越多。虽然 o3/o4-mini 可以凭借其集成的推理流程处理这种情况，但如果 schema 不够清晰或调用条件不够明确，性能可能会下降。

* 工具幻觉可能随复杂度增加：尤其是 o3，有报告称在工具集庞大且定义不足时，会出现幻觉式或臆测性的工具调用。明确的指令有助于缓解这一问题（例如："Only use tools X, Y, Z. Do not invent tool calls or defer them to future turns."）

归根结底，性能会因用例而异；因此，投入建设你信得过、可用于迭代的评估非常重要。


**问：工具中可以有深层嵌套的参数吗，还是应该把 schema "扁平化"？**

**答：** 同样没有硬性规定。不过，即使你的嵌套结构在技术上受支持，层级很深的参数树也可能影响性能或可靠性。如果拿不准，我们建议尽量让参数保持扁平。

扁平结构通常更便于模型推理：在较扁平的 schema 中，参数字段位于顶层，一目了然。这减少了内部解析和组织结构的需要，有助于避免嵌套对象只填了一部分或字段组合无效等问题。对于深层嵌套的对象，尤其是字段名重复或语义相近的对象，模型更有可能遗漏或误用参数。

嵌套有助于组织复杂逻辑，但需要额外小心：对于天然涉及结构化输入的领域，例如配置负载、复杂的搜索过滤条件或表单提交，嵌套有助于组织相关参数。不过，你必须使用清晰的字段描述、anyOf 逻辑或严格的 schema 等技术，来防止无效的参数组合并提高模型的可靠性

最好的选择方式是用你自己的评估进行测试并衡量效果。不存在"放之四海而皆准"的方案，因为调用行为是涌现出来的，并且对提示很敏感


**问：这份函数调用指导是否适用于自定义工具格式？**

**答：** 不能保证。本文档中的指导假定你使用标准的 `tools` 模型参数来传递函数 schema，如我们关于函数调用的[通用指南](https://platform.openai.com/docs/guides/function-calling)中所示。我们的 o3/o4-mini 模型经过训练，能够原生理解并使用这些 schema 来选择工具和构造参数。

如果你改为在开发者编写的提示中用自然语言提供自定义工具定义（例如在开发者消息或用户消息中内联定义工具），本指导可能并不完全适用。在这些情况下：
模型不会依赖其内部对工具 schema 的先验知识。 
你可能需要更明确地提供少样本示例、输出格式和工具选择标准。 
如果没有 schema 层面的锚定，参数构造的可靠性可能会下降。

请尽可能使用结构化的 tools 参数。如果必须用自由文本定义工具，请将其视为一种自定义协议，并进行相应的测试。